# Philadelphia, PA — Land Value Tax Model (Post-Abatement Baseline)

**Policy assumptions:**
- **Levy scope**: Full combined rate (city + school district) — 1.3998% = 13.998 mills applied to 100% of assessed value
- **Reform**: Split-rate 4:1 (land taxed at 4× the improvement millage rate)
- **Pre-shift baseline**: All previously abated properties are taxed on their full assessed building value, as if the 10-year construction abatement has expired for every parcel. `exempt_building` is treated as taxable.
- **Exemptions**: Fully-exempt parcels (homestead-wiped homesteaders, institutional/nonprofit) remain exempt in both baseline and reform.
- **Data source**: Philadelphia Office of Property Assessment (OPA) via Carto public API
- **Revenue target**: Post-abatement baseline total (higher than FY2024 actuals because abated building values are added back)

**Key limitation**: OPA's land/building split defaults to 20% land for ~45% of improved parcels
(especially multi-family and commercial). This understates land value and attenuates the split-rate
impact relative to a market-derived land assessment.


In [1]:
import sys
import io
import urllib.parse
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

CITY_NAME = 'philadelphia_post_abatement'
STATE_FIPS = '42'
COUNTY_FIPS = '101'
MODEL_TYPE = 'split_rate_4to1_post_abatement'
LAND_IMPROVEMENT_RATIO = 4.0
MILLAGE = 13.998  # combined city (0.6317%) + school district (0.7681%) = 1.3998%
# FY2024 city-only collections: ~$796M; school district separate; combined ~$2.1B
# We model the full combined levy — validate against modeled base directly.

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Section 2 — Fetch / Load Parcel Data

**Column mapping**

| Concept | Source | Column | Notes |
|---|---|---|---|
| Land value (taxable) | `assessments` year=2024 | `taxable_land` | Post-exemption; 2024 vintage matches FY2024 billing |
| Improvement value (taxable) | `assessments` year=2024 | `taxable_building` | Post-exemption; 2024 vintage |
| Total assessed value | `assessments` year=2024 | `market_value` | Pre-exemption gross |
| Use/class code | `opa_properties_public` | `category_code` | Integer 1–15 |
| Geometry | `opa_properties_public` | `the_geom` | WKB point (parcel centroid) |

**Why 2024 assessments?**
The current OPA table reflects 2026 reassessments. FY2024 taxes were billed on 2024 assessments.
Using 2026 values overstates the modeled city levy by ~21% vs. actual FY2024 collections.
The Carto `assessments` history table has per-year taxable values; using `year=2024` brings
the city-levy cross-check to within ~5% of actual FY2024 collections. The residual ~5% gap
is current-year delinquency (non-collection), which is ~4–5% historically for Philadelphia.

**Assessment ratio**: 100% full market value (no assessment ratio)

**Millage source**: City of Philadelphia combined rate: 1.3998% = 13.998 mills
(city levy 0.6317% + school district 0.7681%)

In [2]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'

if PARCEL_PATH.exists():
    gdf = gpd.read_parquet(PARCEL_PATH)
    print(f'Loaded {len(gdf):,} parcels from cache')
else:
    # Step 1: Download 2024 assessment-year taxable values (vintage used for FY2024 billing)
    print('Downloading 2024 assessment-year taxable values from Carto...')
    q_assess = (
        'SELECT parcel_number, taxable_land, taxable_building, market_value, '
        'exempt_land, exempt_building FROM assessments WHERE year = 2024'
    )
    url_assess = f'https://phl.carto.com/api/v2/sql?q={urllib.parse.quote(q_assess)}&format=csv'
    r_assess = requests.get(url_assess, timeout=300)
    r_assess.raise_for_status()
    assessments = pd.read_csv(io.StringIO(r_assess.text), low_memory=False)
    assessments['parcel_number'] = assessments['parcel_number'].astype(str)
    print(f'  {len(assessments):,} parcels in 2024 assessment year')

    # Step 2: Download current OPA for geometry and category codes
    print('Downloading OPA properties for geometry and category codes...')
    q_opa = 'SELECT parcel_number, pin, category_code, total_area, the_geom FROM opa_properties_public'
    url_opa = f'https://phl.carto.com/api/v2/sql?q={urllib.parse.quote(q_opa)}&format=csv'
    r_opa = requests.get(url_opa, timeout=300)
    r_opa.raise_for_status()
    opa_geo = pd.read_csv(io.StringIO(r_opa.text), low_memory=False)
    opa_geo['parcel_number'] = opa_geo['parcel_number'].astype(str)
    print(f'  {len(opa_geo):,} parcels in current OPA')

    # Step 3: Join assessments to OPA geometry on parcel_number
    merged = assessments.merge(opa_geo, on='parcel_number', how='inner')
    print(f'  {len(merged):,} parcels after join')

    # Step 4: Parse WKB geometry (point centroids)
    gdf = gpd.GeoDataFrame(
        merged,
        geometry=gpd.GeoSeries.from_wkb(merged['the_geom'], crs='EPSG:4326'),
        crs='EPSG:4326',
    )
    gdf = gdf.drop(columns=['the_geom'], errors='ignore')
    gdf = gdf[gdf['geometry'].notna()].copy()
    print(f'  {len(gdf):,} parcels with geometry')

    for col in ['taxable_land', 'taxable_building', 'market_value', 'exempt_land', 'exempt_building']:
        gdf[col] = pd.to_numeric(gdf[col], errors='coerce').fillna(0.0)

    gdf.to_parquet(PARCEL_PATH)
    print(f'  Cached to {PARCEL_PATH}')

print(f'Columns: {list(gdf.columns)}')
print(gdf[['taxable_land', 'taxable_building', 'market_value']].describe())

Loaded 579,814 parcels from cache
Columns: ['parcel_number', 'taxable_land', 'taxable_building', 'market_value', 'exempt_land', 'exempt_building', 'pin', 'category_code', 'total_area', 'geometry']
       taxable_land  taxable_building  market_value
count  5.798140e+05      5.798140e+05  5.798140e+05
mean   6.316223e+04      1.649179e+05  3.503924e+05
std    4.729710e+05      1.528638e+06  3.054850e+06
min    0.000000e+00      0.000000e+00  0.000000e+00
25%    2.030000e+04      3.120000e+04  1.007000e+05
50%    3.490000e+04      9.216000e+04  1.772000e+05
75%    5.284000e+04      1.630400e+05  2.711000e+05
max    7.800000e+07      3.120000e+08  4.962848e+08


## Section 3 — Classify and Validate

In [3]:
# Coerce category_code to string for mapping
gdf['category_code'] = (
    pd.to_numeric(gdf['category_code'], errors='coerce')
    .astype('Int64')
    .astype(str)
)

CATEGORY_MAP = {
    '1':  'Single Family Residential',
    '2':  'Small Multi-Family (2-4 units)',
    '3':  'Mixed Use',
    '4':  'Commercial',
    '5':  'Industrial',
    '6':  'Vacant Land',
    '7':  'Other Commercial',
    '8':  'Other Residential',
    '9':  'Hotel',
    '10': 'Office / Commercial Condo',
    '11': 'Other',
    '12': 'Vacant Land',
    '13': 'Vacant Land',
    '14': 'Large Multi-Family (5+ units)',
    '15': 'Retail / General Commercial',
}
gdf['PROPERTY_CATEGORY'] = gdf['category_code'].map(CATEGORY_MAP).fillna('Other')

# Override 1: $0 improvement → Vacant Land
gdf.loc[gdf['taxable_building'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

# Override 2: separate abated improved parcels from genuinely vacant land.
GENUINE_VACANT_CODES = {'6', '12', '13'}
abated_mask = (
    (gdf['taxable_building'] <= 0) &
    (gdf['taxable_land'] > 0) &
    (~gdf['category_code'].isin(GENUINE_VACANT_CODES))
)
gdf.loc[abated_mask, 'PROPERTY_CATEGORY'] = 'Abated / Construction Exemption'

# Override 3 (NEW): OPA classifies ~1,483 parcels as vacant (codes 6/12/13) but records
# a non-zero taxable_building — small structures on lots that were never recategorized.
# Their improvement value makes them behave like improved parcels, not vacant land.
# Move them to "Improved Vacant Land" so they don't distort the Vacant Land result.
improved_vacant_mask = (
    gdf['category_code'].isin(GENUINE_VACANT_CODES) &
    (gdf['taxable_building'] > 0)
)
gdf.loc[improved_vacant_mask, 'PROPERTY_CATEGORY'] = 'Improved Vacant Land'

# Full exemption flag: both taxable values are zero
gdf['taxable_total'] = (gdf['taxable_land'] + gdf['taxable_building']).clip(lower=0)
gdf['full_exmp'] = (gdf['taxable_total'] <= 0).astype(int)

# Override 4: re-classify fully exempt parcels by their OPA type with "— Exempt" suffix.
EXEMPT_CATEGORY_MAP = {k: v + ' — Exempt' for k, v in CATEGORY_MAP.items()}
exempt_mask = gdf['full_exmp'] == 1
gdf.loc[exempt_mask, 'PROPERTY_CATEGORY'] = (
    gdf.loc[exempt_mask, 'category_code']
    .map(EXEMPT_CATEGORY_MAP)
    .fillna('Other — Exempt')
)

print(f'Total parcels: {len(gdf):,}')
print(f'Fully exempt: {gdf["full_exmp"].sum():,}  |  '
      f'Abated: {abated_mask.sum():,}  |  '
      f'Improved vacant: {improved_vacant_mask.sum():,}  |  '
      f'Taxable: {(gdf["full_exmp"] == 0).sum():,}')
print()
print('Property category distribution:')
print(gdf['PROPERTY_CATEGORY'].value_counts().to_string())

Total parcels: 579,814
Fully exempt: 44,116  |  Abated: 30,111  |  Improved vacant: 1,489  |  Taxable: 535,698

Property category distribution:
PROPERTY_CATEGORY
Single Family Residential                  408137
Small Multi-Family (2-4 units)              37944
Abated / Construction Exemption             30111
Vacant Land                                 27925
Single Family Residential — Exempt          27059
Mixed Use                                   13399
Vacant Land — Exempt                        11860
Commercial                                   8523
Industrial                                   3485
Commercial — Exempt                          3280
Large Multi-Family (5+ units)                2547
Improved Vacant Land                         1489
Small Multi-Family (2-4 units) — Exempt      1114
Other Residential                            1068
Office / Commercial Condo                     805
Large Multi-Family (5+ units) — Exempt        282
Industrial — Exempt                   

## Section 4 — Post-Abatement Baseline Tax

Philadelphia's combined real estate tax rate is **1.3998% = 13.998 mills** applied to 100% of assessed value.

In this model the pre-shift baseline treats all active construction abatements as **expired**.
Abated parcels have `taxable_building = 0` in OPA, but OPA still carries the full assessed building
value in `exempt_building`. We restore that value so every improved parcel contributes its full
building assessment to the baseline.

- **Non-abated improved parcels**: `taxable_land + taxable_building` (unchanged)
- **Abated parcels**: `taxable_land + exempt_building` (fallback: `market_value − taxable_land` for ~2,100 mid-construction parcels where `exempt_building = 0`)
- **Fully exempt**: $0 (homestead-wiped homesteaders, institutional)

The resulting baseline revenue is **higher than FY2024 actuals** by the amount of abated building value added back.


In [4]:
gdf['millage_rate'] = MILLAGE

# Post-abatement baseline: restore abated building values to the taxable base.
gdf['model_land']     = pd.to_numeric(gdf['taxable_land'],     errors='coerce').fillna(0).clip(lower=0)
gdf['model_building'] = pd.to_numeric(gdf['taxable_building'], errors='coerce').fillna(0).clip(lower=0)

abated = gdf['PROPERTY_CATEGORY'] == 'Abated / Construction Exemption'
exempt_bldg  = pd.to_numeric(gdf['exempt_building'], errors='coerce').fillna(0)
market_val   = pd.to_numeric(gdf['market_value'],    errors='coerce').fillna(0)
tax_land     = pd.to_numeric(gdf['taxable_land'],     errors='coerce').fillna(0)
implied_bldg = (market_val - tax_land).clip(lower=0)
abated_bldg  = exempt_bldg.where(exempt_bldg > 0, implied_bldg)
gdf.loc[abated, 'model_building'] = abated_bldg[abated].values

n_exempt_bldg = int((abated & (exempt_bldg > 0)).sum())
n_fallback    = int((abated & (exempt_bldg <= 0)).sum())
abated_bldg_total = gdf.loc[abated, 'model_building'].sum()
print(f'Abated parcels using exempt_building:        {n_exempt_bldg:,}')
print(f'Abated parcels using market_value fallback:  {n_fallback:,}')
print(f'Total abated building base restored:         ${abated_bldg_total/1e9:.2f}B')
print()

# post_abatement_total: hypothetical fully-taxable base (abatements treated as expired)
gdf['post_abatement_total'] = (gdf['model_land'] + gdf['model_building']).clip(lower=0)

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='post_abatement_total',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

actual_fy2024 = gdf['taxable_total'].mul(MILLAGE / 1000).sum()
print(f'Post-abatement baseline (city + school): ${current_revenue:,.0f}')
print(f'Actual FY2024 combined levy (modeled):   ${actual_fy2024:,.0f}')
print(f'Added by restoring abated buildings:     ${current_revenue - actual_fy2024:,.0f}')


Abated parcels using exempt_building:        27,984
Abated parcels using market_value fallback:  2,127
Total abated building base restored:         $15.00B

Post-abatement baseline (city + school): $2,061,178,704
Actual FY2024 combined levy (modeled):   $1,851,152,567
Added by restoring abated buildings:     $210,026,137


## Section 5 — Split-Rate Model (4:1, Post-Abatement Baseline)


In [5]:
# model_land and model_building already set in Section 4 (post-abatement baseline).

taxable = gdf[gdf['full_exmp'] == 0].copy()

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='model_land',
    improvement_value_col='model_building',
    current_revenue=taxable['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

exempt = gdf[gdf['full_exmp'] == 1].copy()
exempt['new_tax'] = 0.0
exempt['tax_change'] = 0.0
exempt['tax_change_pct'] = 0.0
exempt['taxable_land_value'] = 0.0
exempt['taxable_improvement_value'] = 0.0
gdf = pd.concat([taxable, exempt]).sort_index()

print(f'Land millage:        {land_millage:.4f} mills')
print(f'Improvement millage: {improvement_millage:.4f} mills')
print(f'Revenue check:       ${new_revenue:,.0f} (target: ${taxable["current_tax"].sum():,.0f})')
print()

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title='Philadelphia — 4:1 Split-Rate Tax Impact (OPA Land Values, Post-Abatement Baseline)')


Land millage:        32.0662 mills
Improvement millage: 8.0166 mills
Revenue check:       $2,061,178,704 (target: $2,061,178,704)


Philadelphia — 4:1 Split-Rate Tax Impact (OPA Land Values, Post-Abatement Baseline)
                               Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
              Single Family Residential 408137    $-17,263,283       -1.7%       $-42         $-65    5.8%      -8.3%            28.0%             5.9%
         Small Multi-Family (2-4 units)  37944     $-4,260,555       -1.9%      $-112        $-250   -3.2%      -8.4%             8.4%             2.8%
        Abated / Construction Exemption  30111    $-33,287,528      -13.1%    $-1,105        $-359   -7.6%     -14.8%             7.2%            59.6%
                            Vacant Land  27925     $43,955,545      129.1%     $1,574         $506  129.1%     129.1%           100.0%             0.0%
     Single Family Resid

## Validation Summary

| Check | Result |
|---|---|
| Pre-shift baseline | Post-abatement: abated building values restored via `exempt_building` (+ fallback) |
| Revenue target | Post-abatement baseline; **higher than FY2024 actuals** by the abated building value added back |
| Assessment vintage | 2024 (matches FY2024 billing) |
| Fully exempt parcels | Remain $0 in both baseline and reform |
| Land/improvement split | OPA official values; ~45% of improved parcels default to 20% land fraction |
| Millage rate | 13.998 mills (city 0.6317% + school 0.7681% = 1.3998%) |


In [6]:
# Census join — must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f'Census join: {gdf["std_geoid"].notna().mean()*100:.1f}% matched')
        except concurrent.futures.TimeoutError:
            print('Census API timed out — skipping census join')
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f'Census join failed: {e}')
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

Census join: 100.0% matched


In [7]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report — 7 PNGs in analysis/reports/philadelphia/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)
print('Done.')

  [warn] philadelphia_post_abatement: non-standard property categories (will be preserved): ['Abated / Construction Exemption', 'Commercial — Exempt', 'Hotel — Exempt', 'Improved Vacant Land', 'Industrial — Exempt', 'Large Multi-Family (5+ units) — Exempt', 'Mixed Use — Exempt', 'Office / Commercial Condo — Exempt', 'Other Commercial — Exempt', 'Other Residential — Exempt', 'Other — Exempt', 'Single Family Residential — Exempt', 'Small Multi-Family (2-4 units) — Exempt', 'Vacant Land — Exempt']


  ✓ philadelphia_post_abatement: 579,814 rows → ../../analysis/data/philadelphia_post_abatement.csv  [model: split_rate_4to1_post_abatement]


Done.
